# 4.4 A count, before and after an event

[04.2](04.2-long-tail.ipynb) fitted one family to one population; [04.3](04.3-penguins.ipynb)
showed what a fit does when the population is several groups. Both described a situation as
it is. This notebook answers the question the lesson was built towards: **did an event
change the process, and if so, which parameter moved?** It assumes the fitter and its two
winners from 04.2, the pooled-versus-grouped lesson of 04.3, and 03.1's release-day flag.

Everything so far described one situation. The move that turns a fit into evidence is to fit
the same family on two sides of an event you already know about, and read what changed. The
event here is the one 03.1 used: Ubuntu releases, ten Thursdays between 2013 and 2017, in
the two channels of the `ubuntu_irc_release_hourly` showcase — `#ubuntu` (the big one) and
`#ubuntu-it`. The variable is the simplest count there is: **messages per Thursday**. The
showcase arrives as hourly totals with the release days already flagged, so the counting is
one `GroupAgg` per channel.

In [ ]:
import pandas as pd

from goad_toolkit.analytics import DistributionFitter, FitResult, NullDistribution, fit_table
from goad_toolkit.datatransforms import Filter, GroupAgg, Pipeline
from goad_toolkit.distributions import DistributionRegistry
from goad_toolkit.visualizer import DistPlot, ECDFPlot, HistogramPlot, NullPlot, PlotSettings

from wa_analyzer.data import load_own_chat, load_showcase

fitter = DistributionFitter(DistributionRegistry(), seed=42)

In [ ]:
hourly = load_showcase("ubuntu_irc_release_hourly")


def messages_per_day(channel: str) -> pd.DataFrame:
    """One row per Thursday: total messages that day, and whether it was a release day."""
    return (
        Pipeline()
        .add(Filter, expr=f"channel == '{channel}'")
        .add(GroupAgg, by=["date", "is_release"], column="messages", agg="sum")
        .apply(hourly)
    )


it_days = messages_per_day("#ubuntu-it")
it_ordinary = it_days.loc[~it_days.is_release, "messages"].to_numpy().astype(float)
it_release = it_days.loc[it_days.is_release, "messages"].to_numpy().astype(float)
print(f"#ubuntu-it: {len(it_ordinary)} ordinary Thursdays, {len(it_release)} release days")
print(f"ordinary: mean {it_ordinary.mean():.0f}, variance {it_ordinary.var():.0f}")

## 4.4.1 Which family, on an ordinary Thursday?

Messages per day is a count, so the textbook hypothesis is **Poisson**: independent events
arriving at a steady rate. Poisson makes a checkable promise — its variance equals its mean.
The printout above already breaks that promise by a factor of a few hundred, so the fit
should fail, and *how* it fails is the lesson. Fit every discrete family and draw the two
that matter over the data.

In [ ]:
it_fits = fitter.fit(it_ordinary, discrete=True)
fit_table(it_fits)[["distribution", "params", "log_likelihood", "ks_pvalue", "best_likelihood", "best_ks"]]

In [ ]:
by_name = {f.distribution: f for f in it_fits if isinstance(f, FitResult)}

steady = PlotSettings(
    figsize=(10, 4),
    title="#ubuntu-it, ordinary Thursdays: a steady rate is the wrong hypothesis",
    xlabel="messages per day",
    ylabel="density",
)
host = HistogramPlot(steady)
fig, ax = host.plot(data=it_ordinary, bins=30, color="lightgrey")
host.plot_on(DistPlot(steady), distribution=by_name["poisson"].frozen_dist,
             color="crimson", label="poisson: one steady rate")
_ = host.plot_on(DistPlot(steady), distribution=by_name["nbinom"].frozen_dist,
                 x_range=(0, it_ordinary.max()), color="steelblue", label="nbinom: a rate that wanders")

The Poisson is a needle. With a mean of roughly 560 it allows day-to-day swings of about ±25
messages; the channel swings between under 100 and over 1,500. Its KS p-value in the table is
zero to fifty decimal places, and that rejection is not a failure of the fit — it is the
finding: **there is no steady rate.** Which Thursday it is matters: the year (the channel
shrank across the window), who happened to be around, what broke that week.

The **negative binomial** is the family for exactly that: a Poisson whose rate is itself
drawn at random each day. Its two parameters are a mean and a dispersion — how far the rate
wanders — and it is the top row of the table, with a KS p-value that does not reject it. So
the honest summary of an ordinary Thursday is not "about 560 messages" but "a rate of about
560 that routinely halves or triples". That is reason 1 again: a description that fits.

## 4.4.2 The same family, on release days

Ten release days is a small sample, but the question is small too: does the family hold, and
which parameter moved? Fit it, and put both samples on one bin-free axis.

In [ ]:
release_fits = fitter.fit(it_release, discrete=True)
release_best = next(f for f in release_fits if isinstance(f, FitResult) and f.best_likelihood)
ordinary_best = by_name["nbinom"]

print(f"ordinary Thursday: {ordinary_best.distribution}, mean {ordinary_best.frozen_dist.mean():.0f}, "
      f"sd {ordinary_best.frozen_dist.std():.0f}")
print(f"release day:       {release_best.distribution}, mean {release_best.frozen_dist.mean():.0f}, "
      f"sd {release_best.frozen_dist.std():.0f}")
print(f"rate on a release day: x{it_release.mean() / it_ordinary.mean():.2f}")

fig, ax = ECDFPlot(PlotSettings(figsize=(8, 4), title="#ubuntu-it: ordinary Thursdays vs release days",
                      xlabel="messages per day", ylabel="share of days at or below")).plot(
    data=it_ordinary, compare=it_release, label="ordinary Thursday", compare_label="release day",
)

Same family on both sides, and what moved is the **rate**: a release day runs at roughly
1.4× an ordinary Thursday, with a similar spread around it. The ECDF shows the same thing
without a single bin: the red curve sits to the right of the blue one along most of its
length — release days are shifted, not reshaped. That sentence, *the rate moved and the
shape did not*, is what fitting on both sides buys over comparing two averages.

## 4.4.3 How sure?

A 1.4× lift on ten days, drawn from a distribution that "routinely halves or triples", could
be luck. The check is the one this course keeps coming back to: **what would this number look
like if the label meant nothing?** `NullDistribution` shuffles the `is_release` flag among
the Thursdays, recomputes the rate difference, and does that a thousand times. `NullPlot`
draws the cloud and marks the real value in it.

In [ ]:
def rate_difference(days: pd.DataFrame) -> float:
    """Mean messages on release days minus mean on ordinary Thursdays."""
    means = days.groupby("is_release").messages.mean()
    return means[True] - means[False]


null = NullDistribution(rate_difference, n_iter=2000, seed=4)
it_null = null.run(it_days, label="is_release")

fig, ax = NullPlot(PlotSettings(figsize=(8, 4), title="#ubuntu-it: is a +230 lift more than chance?",
                                xlabel="release − ordinary, messages per day")).plot(result=it_null)
ax.legend()
print(f"two-sided p: {it_null.p_value():.3f}   one-sided p (release days busier): {it_null.p_value('greater'):.3f}")

The observed lift sits in the upper tail of the cloud but not outside it — a two-sided
p-value around 0.07 (one-sided, "busier", about 0.04). Ten days of a rate that wanders this
much is a thin basis, and the number says so.

Now the same three steps on `#ubuntu`, the channel with ten times the traffic.

In [ ]:
ubuntu_days = messages_per_day("#ubuntu")
ubuntu_null = null.run(ubuntu_days, label="is_release")

for name, days, result in [("#ubuntu-it", it_days, it_null), ("#ubuntu", ubuntu_days, ubuntu_null)]:
    ordinary = days.loc[~days.is_release, "messages"]
    lift = days.loc[days.is_release, "messages"].mean() / ordinary.mean()
    print(f"{name:11s} lift x{lift:.2f}   ordinary-day spread sd/mean = {ordinary.std() / ordinary.mean():.2f}"
          f"   two-sided p = {result.p_value():.4f}")

compare = PlotSettings(
    figsize=(13, 4),
    title="The same lift, two verdicts",
    subplot_titles=["#ubuntu-it: rate x1.4, borderline", "#ubuntu: rate x1.5, unmistakable"],
    xlabel="release − ordinary, messages per day",
    ylabel="density",
    max_cols=2,
)
host = NullPlot(compare)
fig, axes = host.create_figure(n_plots=2)
host.plot_on_axes(NullPlot(compare), axes[0], result=it_null)
host.plot_on_axes(NullPlot(compare), axes[1], result=ubuntu_null)
for ax in axes:
    ax.legend()

Both channels show a release lift of about one and a half. In `#ubuntu` it is unmistakable:
the real value stands far outside anything the shuffled labels produce. In `#ubuntu-it` it is
borderline. **The effect is the same size; the evidence is not**, and the printout says why:
an ordinary Thursday in `#ubuntu` wanders about a third of its mean, in `#ubuntu-it` about
two thirds. Many independent people averaging out make days more alike — 04.1 §4.1.2 again, sums of
many contributions — so the same lift stands clear of the day-to-day noise in the big channel
and drowns in it in the small one.

That is the sample-size lesson in its useful form. What you can detect is the effect
*divided by* the spread of the thing you are measuring, and the spread is a property of your
data you can read off the ordinary days before you look at the event. A student who fits the
ordinary Thursdays first knows in advance whether ten release days can settle the question.

## 4.4.4 Your turn — before and after, on your own chat

`load_own_chat()` raises when there is no export yet — a your-turn section without data has
nothing to test. If the next cell errors, run [01.3](../lesson1/01.3-your-own-chat.ipynb)
first and point `current` in `config.toml` at the file it writes.

In [ ]:
own = load_own_chat()
own["timestamp"] = pd.to_datetime(own["timestamp"])

### Think in families first

Before fitting anything, decide what kind of variable you are looking at and what a change in
it would mean — the family tells you which parameter to read:

| variable in your chat | family | what a change means |
|---|---|---|
| messages per day | Poisson if the rate is steady; **nbinom** when it wanders (it will) | the **rate** moved, or the **dispersion** did — busier, or more erratic |
| seconds between messages, within a burst | **exponential** | the **scale** (mean gap) moved — faster or slower back-and-forth |
| messages per day per person | Poisson / nbinom, one per author | *who* changed, not just how much |

A single fit over your whole chat is rarely interesting: "messages per day is over-dispersed"
is true of every chat, because no group keeps a steady rate for years, and "message length
is lognormal" is true of every chat for the reason [04.2](04.2-long-tail.ipynb) gave. The interesting version is the
one §4.4.1–§4.4.3 just did: **an event you know about splits the data, and the fit on each side says
what changed.** 03.3 asked you to name such an event; use it here.

In [ ]:
my_event = None  # >>> Your turn: "YYYY-MM-DD" of an event you KNOW about (from 03.3) <<<

if my_event is None:
    # Not an event, just a way to make the cells below run: the middle of the chat.
    # Replace it. A split you cannot name is a split you cannot interpret.
    my_event = str(own["timestamp"].median().date())
    print(f"no event set; splitting at the midpoint {my_event} so the code runs")

per_day = (
    own.set_index("timestamp").resample("D").size().rename("messages").reset_index()
)
per_day["after"] = per_day["timestamp"] >= pd.Timestamp(my_event, tz=per_day["timestamp"].dt.tz)
print(per_day.groupby("after").messages.agg(["count", "mean", "var"]).round(1))

**Messages per day, both sides.** Fit the discrete families on each side. Read two things
from the parameters, not one: did the *mean* move, and did the *dispersion* — a group can
send the same number of messages in a much more erratic way, and that is a different finding.

In [ ]:
fitter = DistributionFitter(DistributionRegistry(), seed=42)
for label, side in [("before", per_day[~per_day.after]), ("after", per_day[per_day.after])]:
    fits = fitter.fit(side.messages.to_numpy().astype(float), discrete=True)
    best = next(f for f in fits if isinstance(f, FitResult) and f.best_likelihood)
    print(f"{label:7s} {best.distribution:8s} mean {best.frozen_dist.mean():6.1f}   "
          f"sd {best.frozen_dist.std():6.1f}   ({len(side)} days)")

In [ ]:
own_null = NullDistribution(
    lambda days: days.groupby("after").messages.mean().diff().iloc[-1], n_iter=2000, seed=4,
).run(per_day, label="after")

fig, ax = NullPlot(PlotSettings(figsize=(8, 4), title=f"Messages per day: after {my_event} minus before",
                                xlabel="after − before, messages per day")).plot(result=own_null)
_ = ax.legend()

**Gaps between messages, both sides.** The exponential is the family for waiting times at a
steady rate, and "within a burst" is the only place a chat has anything like a steady rate —
so the gaps are cut at an hour, and the two fits are compared on their *scale*, the mean gap.

⚠️ This one needs second-resolution timestamps. IRC logs `[HH:MM]`, and over half of
consecutive IRC messages land in the same minute, so the gap is zero by construction — a
measurement problem, not a modelling one. WhatsApp exports carry seconds; the check below
refuses to fit if yours does not.

In [ ]:
ts = own["timestamp"].sort_values()
on_the_minute = ((ts.dt.second == 0) & (ts.dt.microsecond == 0)).mean()
if on_the_minute > 0.3:
    print(f"{on_the_minute:.0%} of timestamps land exactly on the minute: too coarse for a gap fit.")
else:
    gaps = pd.DataFrame({"gap": ts.diff().dt.total_seconds(), "timestamp": ts}).dropna()
    gaps = gaps[(gaps.gap > 0) & (gaps.gap < 3600)]  # within an hour: a burst, not overnight
    gaps["after"] = gaps.timestamp >= pd.Timestamp(my_event, tz=gaps.timestamp.dt.tz)
    for label, side in [("before", gaps[~gaps.after]), ("after", gaps[gaps.after])]:
        fit = fitter.fit_distribution("exponential", side.gap.to_numpy())
        print(f"{label:7s} mean gap {fit.frozen_dist.mean():6.0f} s   ({len(side):,} gaps)")  # ty: ignore[unresolved-attribute]

### What to write down

1. **The event and what you expected it to change** — rate, dispersion, or gap scale — before
   you ran the cells. Quote yourself.
2. **The two fits**, as a sentence with parameters in it: "before, nbinom with mean 31 and sd
   19; after, mean 52 and sd 40". A table is fine; a screenshot of a histogram is not.
3. **The null plot, and where the real value sits in it.** If it sits inside the cloud, say so
   — a change you cannot distinguish from chance is a finding about your sample size, and
   §4.4.3 showed how to know that in advance from the spread of the ordinary days.

This is what the `goad` MCP's `goad_analysis_checklist` is built to interview you about; a
before/after question with a named event is exactly the shape it expects.

## Reflection

1. Pick one family from [04.1](04.1-families.ipynb) and name a variable in your own chat you would expect to
   follow it. What would have to be true about *how that variable is generated* for the
   expectation to hold — and what would make the fit fail the way Poisson failed in §4.4.1?
2. In [04.2](04.2-long-tail.ipynb) the log-transform turned an "outlier" into an ordinary observation. Where in your
   own data would the reverse happen — a log making an ordinary point look suspicious?
3. §4.4.3 found the same lift with two different verdicts. For your own event: from the spread
   of the ordinary days alone, how many event days would you have needed to be sure?
4. Your event splits the chat in time; [04.3](04.3-penguins.ipynb) split by group. If the
   people in your chat changed *who was around* at the event, which of the two splits is
   the finding really about?

---

**Where this goes next.** Every fit in this lesson so far was to data as it arrived. The most
useful thing to fit a distribution to is what a *model* left behind:
[04.5-fitting-the-residual](04.5-fitting-the-residual.ipynb) fits a line, subtracts it, and
asks the residual whether the model is done or is missing a mechanism.